# RL-LLM Toolkit Quickstart

This notebook demonstrates the core features of RL-LLM Toolkit with interactive examples.

## Installation

```bash
pip install rl-llm-toolkit
```

In [ ]:
import sys
sys.path.insert(0, '../..')

from rl_llm_toolkit import RLEnvironment, PPOAgent
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output

print("✅ Imports successful!")

## 1. Create Environment

Let's start with the classic CartPole environment.

In [ ]:
env = RLEnvironment("CartPole-v1")

print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Actions: 0 = Left, 1 = Right")

## 2. Test Random Agent

Before training, let's see how a random agent performs.

In [ ]:
def test_random_agent(episodes=5):
    rewards = []
    
    for ep in range(episodes):
        obs, _ = env.reset(seed=42+ep)
        done = False
        episode_reward = 0
        
        while not done:
            action = env.action_space.sample()
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
        
        rewards.append(episode_reward)
        print(f"Episode {ep+1}: Reward = {episode_reward}")
    
    print(f"\nRandom Agent - Mean Reward: {np.mean(rewards):.2f} ± {np.std(rewards):.2f}")
    return rewards

random_rewards = test_random_agent()

## 3. Train PPO Agent

Now let's train a PPO agent to solve the task.

In [ ]:
agent = PPOAgent(
    env=env,
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    seed=42,
)

print("Agent created! Starting training...")

In [ ]:
results = agent.train(
    total_timesteps=50000,
    log_interval=5,
    eval_interval=10000,
    eval_episodes=5,
    progress_bar=True,
)

print(f"\n✅ Training complete!")
print(f"Total episodes: {results['episodes']}")
print(f"Total timesteps: {results['total_timesteps']}")

## 4. Visualize Training Progress

In [ ]:
stats = agent.get_training_stats()
episode_rewards = stats['stats']['episode_rewards']
episode_lengths = stats['stats']['episode_lengths']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Plot rewards
ax1.plot(episode_rewards, alpha=0.3, label='Raw')
window = 10
if len(episode_rewards) >= window:
    smoothed = np.convolve(episode_rewards, np.ones(window)/window, mode='valid')
    ax1.plot(range(window-1, len(episode_rewards)), smoothed, label=f'Smoothed ({window} ep)', linewidth=2)
ax1.set_xlabel('Episode')
ax1.set_ylabel('Reward')
ax1.set_title('Training Rewards')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot episode lengths
ax2.plot(episode_lengths, alpha=0.3, label='Raw')
if len(episode_lengths) >= window:
    smoothed_len = np.convolve(episode_lengths, np.ones(window)/window, mode='valid')
    ax2.plot(range(window-1, len(episode_lengths)), smoothed_len, label=f'Smoothed ({window} ep)', linewidth=2)
ax2.set_xlabel('Episode')
ax2.set_ylabel('Length')
ax2.set_title('Episode Lengths')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Final 10 episodes mean reward: {np.mean(episode_rewards[-10:]):.2f}")

## 5. Evaluate Trained Agent

In [ ]:
eval_results = agent.evaluate(episodes=20, deterministic=True)

print("Evaluation Results:")
print(f"  Mean Reward: {eval_results['mean_reward']:.2f} ± {eval_results['std_reward']:.2f}")
print(f"  Min Reward: {eval_results['min_reward']:.2f}")
print(f"  Max Reward: {eval_results['max_reward']:.2f}")
print(f"  Mean Length: {eval_results['mean_length']:.2f}")

# Compare with random agent
improvement = eval_results['mean_reward'] - np.mean(random_rewards)
print(f"\n📈 Improvement over random: +{improvement:.2f} ({improvement/np.mean(random_rewards)*100:.1f}%)")

## 6. Save and Load Model

In [ ]:
from pathlib import Path

model_path = Path("./models/cartpole_quickstart.pt")
model_path.parent.mkdir(parents=True, exist_ok=True)

agent.save(model_path)
print(f"✅ Model saved to {model_path}")

# Load and verify
new_agent = PPOAgent(env=env)
new_agent.load(model_path)
print("✅ Model loaded successfully")

# Quick test
test_results = new_agent.evaluate(episodes=5, deterministic=True)
print(f"Loaded model mean reward: {test_results['mean_reward']:.2f}")

## 7. Interactive Visualization

Watch the trained agent in action!

In [ ]:
def visualize_episode(agent, max_steps=500):
    obs, _ = env.reset(seed=42)
    done = False
    step = 0
    total_reward = 0
    
    states = []
    actions = []
    rewards = []
    
    while not done and step < max_steps:
        action, info = agent.predict(obs, deterministic=True)
        
        states.append(obs.copy())
        actions.append(action)
        
        obs, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        rewards.append(reward)
        total_reward += reward
        step += 1
    
    print(f"Episode finished after {step} steps")
    print(f"Total reward: {total_reward}")
    
    # Plot state evolution
    states = np.array(states)
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    labels = ['Cart Position', 'Cart Velocity', 'Pole Angle', 'Pole Angular Velocity']
    for i, (ax, label) in enumerate(zip(axes.flat, labels)):
        ax.plot(states[:, i])
        ax.set_xlabel('Step')
        ax.set_ylabel(label)
        ax.set_title(f'{label} over Time')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    return states, actions, rewards

states, actions, rewards = visualize_episode(agent)

## 8. Next Steps

- Try different environments: `LunarLander-v2`, `MountainCar-v0`
- Experiment with hyperparameters
- Add LLM reward shaping (see notebook 02)
- Try DQN algorithm
- Build custom environments

In [ ]:
env.close()
print("✅ Tutorial complete!")